In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('mortgage_pull_through_messy.csv')

In [3]:
for c in df.columns:
    print(c, df[c].nunique(dropna=False))

loan_number 49360
application_date 7661
lock_date 7835
note_rate 5213
LTV 21834
CLTV 23060
DTI 18913
FICO 329
UW_system 18
funded_date 7938
action_date 6016
file_status 27
loan_program 28


In [4]:
print(df.dtypes)

loan_number         float64
application_date     object
lock_date            object
note_rate            object
LTV                  object
CLTV                 object
DTI                  object
FICO                 object
UW_system            object
funded_date          object
action_date          object
file_status          object
loan_program         object
dtype: object


In [5]:
print(df.columns)

Index(['loan_number', 'application_date', 'lock_date', 'note_rate', 'LTV',
       'CLTV', 'DTI', 'FICO', 'UW_system', 'funded_date', 'action_date',
       'file_status', 'loan_program'],
      dtype='object')


In [6]:
original = df['application_date'].copy()

converted = pd.to_datetime(
    original,
    format = 'mixed', 
    errors = 'coerce'
)

bad_mask = original.notna() * converted.isna()

print(original[bad_mask].value_counts())

application_date
13/15/2024    65
31/31/2025    60
02/30/2025    59
00/12/2026    59
2025-99-01    54
Name: count, dtype: int64


In [7]:
for col in df.columns[df.columns.str.contains('_date')]:
    before_na = df[col].isna().sum()

    converted = pd.to_datetime(
        df[col], 
        format = 'mixed', 
        errors = 'coerce'
        )

    after_na = converted.isna().sum()

    print(col, 
          'original NA', before_na,
          'after conversion', after_na,
          'new failtures', after_na - before_na
          )

    df[col] = converted
    df['application_date_missing'] = df['application_date'].isna().astype(int)

application_date original NA 500 after conversion 797 new failtures 297
lock_date original NA 500 after conversion 500 new failtures 0
funded_date original NA 12022 after conversion 12022 new failtures 0
action_date original NA 38466 after conversion 38466 new failtures 0


In [8]:
print(df['note_rate'].dtype)

print(df['note_rate'].head(20))

print(df['note_rate'].value_counts(dropna = False).head(30))

object
0       4.03
1       7.7 
2       6.73
3     6.670%
4       4.74
5      6.09%
6     0.0662
7     0.0648
8       7.48
9       6.64
10    0.0587
11    6.438%
12       NaN
13     5.07%
14      7.69
15      6.65
16    5.038%
17      5.14
18     5.44%
19       NaN
Name: note_rate, dtype: object
note_rate
NaN      739
 6.1     357
 6.3     351
 6.0     325
 6.2     321
 6.6     318
 6.4     314
 6.5     308
 5.9     304
 5.7     295
 6.7     287
 5.8     286
 6.8     277
 5.5     276
 7.0     271
 6.9     267
 5.6     265
 5.4     242
 5.3     214
 7.1     210
 7.2     206
 7.3     202
 5.2     182
 5.1     177
 7.4     162
 7.5     149
 5.0     145
 4.9     135
 7.7     125
 4.8     123
Name: count, dtype: int64


In [9]:
def clean_note_rate(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip().lower()

    if x == 'six point five':
        return 6.5

    if x == '-3%':
        return 3.0

    if x == '65%':
        return 6.5

    if '%' in x:
        try:
            return float(x.replace('%', ''))
        except ValueError:
            return np.nan

    try:
        x = float(x)
    except ValueError:
        return np.nan

    if 0 < x < 1:
        x *= 100

    return x


df['note_rate_clean'] = df['note_rate'].apply(clean_note_rate)

print(df[['note_rate', 'note_rate_clean']].sample(30))

print(df['note_rate_clean'].describe())

print(df['note_rate_clean'].sort_values().head(20))

print(df['note_rate_clean'].sort_values(ascending = False).head(20))

      note_rate  note_rate_clean
45686      6.09            6.090
35571      5.89            5.890
1214      6.26%            6.260
25933      7.2             7.200
44537    0.0517            5.170
18188      7.49            7.490
1870       6.25            6.250
16807      6.62            6.620
5394       5.4             5.400
20614      8.7             8.700
41121       NaN              NaN
4234       6.46            6.460
34573     5.36%            5.360
48419      5.16            5.160
40586     5.10%            5.100
42288    0.0458            4.580
33664    5.612%            5.612
12558    5.638%            5.638
6813      5.87%            5.870
6806       5.3             5.300
34911    0.0826            8.260
46386      6.61            6.610
44877     6.39%            6.390
18390    6.258%            6.258
37315     5.44%            5.440
17988    5.800%            5.800
13140      6.1             6.100
22887    7.058%            7.058
14005    0.0547            5.470
13268     

In [10]:
def clean_percent(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip().lower()

    text_map = {
        'forty five': 45
    }

    if x in text_map:
        return text_map[x]

    x = x.replace('%', '')

    try:
        x = float(x)
    except ValueError:
        return np.nan

    if x in [999, 9999]:
        return np.nan

    if x < 0:
        x = abs(x)

    if 0 < x < 2:
        x *= 100

    return x


for col in ['LTV', 'CLTV', 'DTI']:
    df[f'{col}_clean'] = df[col].apply(clean_percent)

In [11]:
print(df.loc[df['LTV'] == '105%', ['LTV', 'LTV_clean']].head())

print(df.loc[df['LTV'].astype(str) == '999', ['LTV', 'LTV_clean']].head())

print(df.loc[df['CLTV'] == '90%', ['CLTV', 'CLTV_clean']].head())

Empty DataFrame
Columns: [LTV, LTV_clean]
Index: []
      LTV  LTV_clean
70    999        NaN
171   999        NaN
289   999        NaN
410   999        NaN
1043  999        NaN
Empty DataFrame
Columns: [CLTV, CLTV_clean]
Index: []


In [12]:
print(
    df.loc[
        df['LTV'].astype(str).str.contains('105', na=False),
        ['LTV', 'LTV_clean']
    ].head(20)
)

print(
    df.loc[
        df['CLTV'].astype(str).str.contains('90', na=False),
        ['CLTV', 'CLTV_clean']
    ].head(20)
)

           LTV  LTV_clean
28       105.0      105.0
190      105.0      105.0
226      105.0      105.0
271      105.0      105.0
306   105.000%      105.0
363      105.0      105.0
414    105.00%      105.0
422      105.0      105.0
499    105.00%      105.0
566    105.00%      105.0
641    105.00%      105.0
643    105.00%      105.0
656      105.0      105.0
657   105.000%      105.0
660    105.00%      105.0
714      105.0      105.0
740    105.00%      105.0
749      105.0      105.0
937      105.0      105.0
1032     105.0      105.0
        CLTV  CLTV_clean
13    72.90%      72.900
52   90.029%      90.029
74     90.04      90.040
88   72.790%      72.790
116   0.9903      99.030
141   90.68%      90.680
145    90.06      90.060
182    90.3       90.300
206    90.42      90.420
238    90.26      90.260
277    90.4       90.400
299   90.76%      90.760
310   0.6907      69.070
476  90.364%      90.364
554     90.5      90.500
673    90.15      90.150
683    90.4       90.400
712 

In [13]:
print(
    df.loc[
        df['LTV'].astype(str).str.strip() == '999',
        ['LTV', 'LTV_clean']
    ].head(20)
)

print(
    df.loc[
        df['LTV'].astype(str).str.strip() == '999',
        ['LTV', 'LTV_clean']
    ].shape
)

      LTV  LTV_clean
70    999        NaN
171   999        NaN
289   999        NaN
410   999        NaN
1043  999        NaN
1191  999        NaN
1560  999        NaN
2681  999        NaN
2954  999        NaN
3245  999        NaN
3389  999        NaN
3466  999        NaN
3542  999        NaN
5043  999        NaN
5044  999        NaN
5420  999        NaN
6681  999        NaN
6703  999        NaN
8190  999        NaN
8499  999        NaN
(88, 2)


In [14]:
print(df.dtypes)

loan_number                        float64
application_date            datetime64[ns]
lock_date                   datetime64[ns]
note_rate                           object
LTV                                 object
CLTV                                object
DTI                                 object
FICO                                object
UW_system                           object
funded_date                 datetime64[ns]
action_date                 datetime64[ns]
file_status                         object
loan_program                        object
application_date_missing             int64
note_rate_clean                    float64
LTV_clean                          float64
CLTV_clean                         float64
DTI_clean                          float64
dtype: object


In [15]:
fico_test = pd.to_numeric(df['FICO'], errors='coerce')

def clean_fico(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip().lower()

    if x == 'seven hundred':
        return 700

    try:
        x = float(x)
    except ValueError:
        return np.nan

    if x == 999:
        return np.nan

    return x


df['FICO_clean'] = df['FICO'].apply(clean_fico)

In [16]:
print(df['FICO_clean'].describe())
print('Missing:', df['FICO_clean'].isna().sum())

print('\nLowest:')
print(df['FICO_clean'].sort_values().head(20))

print('\nHighest:')
print(df['FICO_clean'].sort_values(ascending=False).head(20))

count    49092.000000
mean       729.085656
std         55.772941
min        420.000000
25%        693.000000
50%        730.000000
75%        767.000000
max        850.000000
Name: FICO_clean, dtype: float64
Missing: 908

Lowest:
36243    420.0
30674    420.0
8571     420.0
3854     420.0
27340    420.0
16317    420.0
43012    420.0
30526    420.0
34444    420.0
37665    420.0
9695     420.0
34240    420.0
20867    420.0
4218     420.0
10707    420.0
47762    420.0
3731     420.0
42481    420.0
16480    420.0
35428    420.0
Name: FICO_clean, dtype: float64

Highest:
4550     850.0
49439    850.0
29548    850.0
10604    850.0
22185    850.0
43719    850.0
17129    850.0
43724    850.0
709      850.0
29459    850.0
10686    850.0
29399    850.0
47750    850.0
29383    850.0
11923    850.0
2659     850.0
22260    850.0
29250    850.0
22264    850.0
46395    850.0
Name: FICO_clean, dtype: float64


In [17]:
print(df['UW_system'].unique().tolist())

['D.U.', ' DU ', 'Desktop Underwriter', 'LPA', 'DESKTOP UNDERWRITER', 'LP', ' LPA ', 'du', 'Manual', 'DU', 'lpa', 'manual underwriting', 'Loan Product Advisor', nan, 'Loan Prospector', 'Man UW', 'manual', 'MANUAL UW']


In [18]:
uw_cols_uniques = [
    'D.U.', ' DU ', 'Desktop Underwriter',
    'LPA', 'DESKTOP UNDERWRITER', 'LP',
    ' LPA ', 'du', 'Manual',
    'DU', 'lpa', 'manual underwriting',
    'Loan Product Advisor', 'Loan Prospector',
    'Man UW', 'manual', 'MANUAL UW']

uw_cols_key = ['DU', 'DU', 'DU',
               'LP', 'DU', 'LP',
               'LP', 'DU', 'Manual',
               'DU', 'LP', 'Manual',
               'LP', 'LP',
               'Manual', 'Manual', 'Manual'
               ]


uw_map = dict(zip(uw_cols_uniques, uw_cols_key))

df['uw_clean'] = df['UW_system'].map(uw_map)

In [19]:
print(
    df.loc[df['uw_clean'].isna(), 'UW_system']
      .value_counts(dropna=False)
)

UW_system
NaN    500
Name: count, dtype: int64


In [20]:
print(df['file_status'].unique().tolist())

['Closed/Funded', nan, 'Declined', 'Withdrawn', ' Funded ', 'funded', 'FUNDED', 'Fund', 'Funded', 'CANCELLED', 'WD', 'W/D', 'Lock Expired', 'cancel', 'withdrawn', 'Withdraw', 'Cancelled', 'Credit Denied', 'Denied', 'DENY', 'Canceled', 'Expired', 'CXL', ' WITHDRAWN ', 'expired', 'denied', 'EXP']


In [21]:
loan_status_uniquies = ['Closed/Funded', 'Declined', 
                        'Withdrawn', ' Funded ', 'funded', 
                        'FUNDED', 'Fund', 'Funded', 
                        'CANCELLED', 'WD', 'W/D', 
                        'Lock Expired', 'cancel', 'withdrawn', 
                        'Withdraw', 'Cancelled', 'Credit Denied', 
                        'Denied', 'DENY', 'Canceled', 
                        'Expired', 'CXL', ' WITHDRAWN ', 
                        'expired', 'denied', 'EXP']

loan_status_key = ['Funded', 'Dead',
                   'Dead', 'Funded', 'Funded',
                   'Funded', 'Funded', 'Funded',
                   'Dead', 'Dead', 'Dead',
                   'Dead', 'Dead', 'Dead',
                   'Dead', 'Dead', 'Dead', 
                   'Dead', 'Dead', 'Dead', 
                   'Dead', 'Dead', 'Dead', 
                   'Dead', 'Dead', 'Dead', 
                   ]

ls_map = dict(zip(loan_status_uniquies, loan_status_key))

df['ls_clean'] = df['file_status'].map(ls_map)

In [22]:
print(df['loan_program'].unique().tolist())

['CONV30', 'Conv 30yr', 'Conv 30', '30yr fixed', '30 Year VA', 'FHA30', 'Non-Conforming 30', 'FHA 30YR', 'JUMBO30', 'Conventional 15YR', 'F.H.A.', 'FHA', 'Conv 15', ' conventional 30 ', 'Conventional 30YR', 'Jumbo', 'V.A.', 'VA', '30 Year Conventional', 'CONV15', 'VA30', 'VA 30YR', '15yr fixed', nan, '30YR FHA', '15 Year Conventional', '30yr jumbo', 'Jumbo 30YR']


In [23]:
loan_program_unique = ['CONV30', 'Conv 30yr', 'Conv 30', 
                       '30yr fixed', '30 Year VA', 'FHA30', 
                       'Non-Conforming 30', 'FHA 30YR', 'JUMBO30', 
                       'Conventional 15YR', 'F.H.A.', 'FHA', 
                       'Conv 15', ' conventional 30 ', 'Conventional 30YR', 
                       'Jumbo', 'V.A.', 'VA', 
                       '30 Year Conventional', 'CONV15', 'VA30', 
                       'VA 30YR', '15yr fixed',
                       '30YR FHA', '15 Year Conventional', '30yr jumbo', 'Jumbo 30YR']


loan_program_key = ['Conventional', 'Conventional', 'Conventional', 
                       'Conventional', 'VA', 'FHA', 
                       'Non_Conforming', 'FHA', 'Non_Conforming', 
                       'Conventional', 'FHA', 'FHA', 
                       'Conventional', ' Conventional', 'Conventional', 
                       'Non_Conforming', 'VA', 'VA', 
                       'Conventional', 'Conventional', 'VA', 
                       'VA', 'Conventional', 
                       'FHA', 'Conventional', 'Non_Conforming', 'Non_Conforming']

loan_program_map = dict(zip(loan_program_unique, loan_program_key))

df['loan_category'] = df['loan_program'].map(loan_program_map)

In [24]:
print(df.dtypes)

loan_number                        float64
application_date            datetime64[ns]
lock_date                   datetime64[ns]
note_rate                           object
LTV                                 object
CLTV                                object
DTI                                 object
FICO                                object
UW_system                           object
funded_date                 datetime64[ns]
action_date                 datetime64[ns]
file_status                         object
loan_program                        object
application_date_missing             int64
note_rate_clean                    float64
LTV_clean                          float64
CLTV_clean                         float64
DTI_clean                          float64
FICO_clean                         float64
uw_clean                            object
ls_clean                            object
loan_category                       object
dtype: object


In [27]:
print(df['ls_clean'].unique())

['Funded' nan 'Dead']


In [31]:
print(df['ls_clean'].isna().sum())
print(sum(df['ls_clean'] == 'Funded'))
print(sum(df['ls_clean'] == 'Dead'))


500
37983
11517


In [35]:
df['dead_lock'] = (
    (df['ls_clean'] == 'Dead') |
    (df['action_date'].notna())
).astype(int)

In [36]:
print(df.dtypes)

loan_number                        float64
application_date            datetime64[ns]
lock_date                   datetime64[ns]
note_rate                           object
LTV                                 object
CLTV                                object
DTI                                 object
FICO                                object
UW_system                           object
funded_date                 datetime64[ns]
action_date                 datetime64[ns]
file_status                         object
loan_program                        object
application_date_missing             int64
note_rate_clean                    float64
LTV_clean                          float64
CLTV_clean                         float64
DTI_clean                          float64
FICO_clean                         float64
uw_clean                            object
ls_clean                            object
loan_category                       object
dead_lock                            int64
dtype: obje

In [37]:
df.to_parquet('pull_through_clean.parquet')